In [1]:
import numpy as np
import pandas as pd
import os

# how many seeds did we do for everything?
N_SEEDS = 100

In [2]:
# load in baselines results + get uniform results
baseline_df = pd.read_csv("logs/gpqa-all/baselines_gpqa_all.csv")
uniform_df = baseline_df.query("policy == 'unif' and f == 'zero'").copy()\
.sort_values(by=["prop_budget", "seed"]).reset_index(drop=True)

# FAQ (yes transfer learning, 0% complete cold start)
faq_transfer_0_df = pd.read_csv("logs/gpqa-all/faq_final_transfer_0.csv").sort_values(by=["prop_budget", "seed"]).reset_index(drop=True)

In [3]:
# the 4 columns that define a baseline setting (aside from seed) + the 1 column that defines a data scenario
variant_cols = list(baseline_df.columns[:4])
scenario_cols = list(baseline_df.columns[:1])

In [4]:
# get the best "Historical" baseline setting per setting, then get per seed results for said setting
baseline_means = baseline_df.groupby(variant_cols, dropna=False).mean().reset_index()
best_baseline_settings = baseline_means.loc[
    baseline_means.groupby(scenario_cols, dropna=False)["mean_width"].idxmin(), variant_cols]
best_baseline_df = pd.merge(baseline_df, best_baseline_settings, how="inner", on=variant_cols)\
.sort_values(by=["prop_budget", "seed"]).reset_index(drop=True)

In [5]:
# get ESS multipliers for best baselines & faq (each dataframe separately) vs. uniform
best_baseline_df["ess_multiplier"] = (uniform_df.mean_width / best_baseline_df.mean_width) ** 2
faq_transfer_0_df["ess_multiplier"] = (uniform_df.mean_width / faq_transfer_0_df.mean_width) ** 2

In [6]:
# FAQ (transfer, 0% data) summary
faq_transfer_0_summary = faq_transfer_0_df[
    scenario_cols + ["seed"] + ["ess_multiplier"]].groupby(scenario_cols, dropna=False).mean().reset_index()
faq_transfer_0_summary["ess_multiplier_serr"] = faq_transfer_0_df[
    scenario_cols + ["seed"] + ["ess_multiplier"]].groupby(
    scenario_cols, dropna=False).std().reset_index().ess_multiplier / np.sqrt(N_SEEDS)
faq_transfer_0_summary["coverage"] = faq_transfer_0_df[
    scenario_cols + ["seed"] + ["coverage"]].groupby(
    scenario_cols, dropna=False).mean().reset_index().coverage
faq_transfer_0_summary["coverage_serr"] = faq_transfer_0_df[
    scenario_cols + ["seed"] + ["coverage"]].groupby(
    scenario_cols, dropna=False).std().reset_index().coverage / np.sqrt(N_SEEDS)
faq_transfer_0_summary["mean_width"] = faq_transfer_0_df[
    scenario_cols + ["seed"] + ["mean_width"]].groupby(
    scenario_cols, dropna=False).mean().reset_index().mean_width
faq_transfer_0_summary["mean_width_serr"] = faq_transfer_0_df[
    scenario_cols + ["seed"] + ["mean_width"]].groupby(
    scenario_cols, dropna=False).std().reset_index().mean_width / np.sqrt(N_SEEDS)

In [7]:
# best baselines summary
best_baseline_summary = best_baseline_df[
    scenario_cols + ["seed"] + ["ess_multiplier"]].groupby(scenario_cols, dropna=False).mean().reset_index()
best_baseline_summary["ess_multiplier_serr"] = best_baseline_df[
    scenario_cols + ["seed"] + ["ess_multiplier"]].groupby(
    scenario_cols, dropna=False).std().reset_index().ess_multiplier / np.sqrt(N_SEEDS)
best_baseline_summary["coverage"] = best_baseline_df[
    scenario_cols + ["seed"] + ["coverage"]].groupby(
    scenario_cols, dropna=False).mean().reset_index().coverage
best_baseline_summary["coverage_serr"] = best_baseline_df[
    scenario_cols + ["seed"] + ["coverage"]].groupby(
    scenario_cols, dropna=False).std().reset_index().coverage / np.sqrt(N_SEEDS)
best_baseline_summary["mean_width"] = best_baseline_df[
    scenario_cols + ["seed"] + ["mean_width"]].groupby(
    scenario_cols, dropna=False).mean().reset_index().mean_width
best_baseline_summary["mean_width_serr"] = best_baseline_df[
    scenario_cols + ["seed"] + ["mean_width"]].groupby(
    scenario_cols, dropna=False).std().reset_index().mean_width / np.sqrt(N_SEEDS)

In [8]:
# best uniform summary
uniform_df["ess_multiplier"] = 1.0
uniform_summary = uniform_df[
    scenario_cols + ["seed"] + ["ess_multiplier"]].groupby(scenario_cols, dropna=False).mean().reset_index()
uniform_summary["ess_multiplier_serr"] = uniform_df[
    scenario_cols + ["seed"] + ["ess_multiplier"]].groupby(
    scenario_cols, dropna=False).std().reset_index().ess_multiplier / np.sqrt(N_SEEDS)
uniform_summary["coverage"] = uniform_df[
    scenario_cols + ["seed"] + ["coverage"]].groupby(
    scenario_cols, dropna=False).mean().reset_index().coverage
uniform_summary["coverage_serr"] = uniform_df[
    scenario_cols + ["seed"] + ["coverage"]].groupby(
    scenario_cols, dropna=False).std().reset_index().coverage / np.sqrt(N_SEEDS)
uniform_summary["mean_width"] = uniform_df[
    scenario_cols + ["seed"] + ["mean_width"]].groupby(
    scenario_cols, dropna=False).mean().reset_index().mean_width
uniform_summary["mean_width_serr"] = uniform_df[
    scenario_cols + ["seed"] + ["mean_width"]].groupby(
    scenario_cols, dropna=False).std().reset_index().mean_width / np.sqrt(N_SEEDS)

In [9]:
# save our summaries to .csv
uniform_summary.to_csv("logs/gpqa-all/uniform_summary.csv", index=False)
faq_transfer_0_summary.to_csv("logs/gpqa-all/faq_transfer_0_summary.csv", index=False)
best_baseline_summary.to_csv("logs/gpqa-all/best_baseline_summary.csv", index=False)